In [12]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = 'sqlite:///mlflow.db'

In [13]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

client.search_experiments()

[<Experiment: artifact_location='/home/dongenxi/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1788169670539, experiment_id='2', last_update_time=1788169670539, lifecycle_stage='active', name='my-cool-experiment', tags={}>,
 <Experiment: artifact_location='/home/dongenxi/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1788146465721, experiment_id='1', last_update_time=1788146465721, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='/home/dongenxi/mlops-zoomcamp/02-experiment-tracking/mlruns/0', creation_time=1788146465692, experiment_id='0', last_update_time=1788146465692, lifecycle_stage='active', name='Default', tags={}>]

In [15]:
# client.create_experiment(name='my-cool-experiment')

In [16]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string='metric.rmse < 7',
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=['metric.rmse ASC']
)

In [17]:
for run in runs:
    print(f'run id: {run.info.run_id}, rmse: {run.data.metrics["rmse"]:.4f}')

run id: 665d3838e16a404e93c419edf868805e, rmse: 6.3184
run id: 7775070067514d7490c04d5f642c070d, rmse: 6.4000
run id: 2c1a507eb3e1496692fe15677cee335b, rmse: 6.7423
run id: f7577e8748774de6b7b2ee7473bcecaf, rmse: 6.9180


In [ ]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# 注册表里有的 run_id不能再加入了
run_id = 'f7577e8748774de6b7b2ee7473bcecaf'
model_uri = f'runs:/{run_id}/model'
mlflow.register_model(model_uri=model_uri, name='nyc-taxi-regressor')

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/08/31 10:54:59 WARNING mlflow.tracking._model_registry.fluent: Run with id f7577e8748774de6b7b2ee7473bcecaf has no artifacts at artifact path 'model', registering model based on models:/m-068c811689434d0f9f469f70039004f2 instead
Created version '5' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1788173699253, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1788173699253, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='f7577e8748774de6b7b2ee7473bcecaf', run_link=None, source='models:/m-068c811689434d0f9f469f70039004f2', status='READY', status_message=None, tags={}, user_id=None, version=5>

In [21]:
model_name = 'nyc-taxi-regressor'
lastest_version = client.get_latest_versions(name=model_name)

for version in lastest_version:
    print(f'version: {version.version}, stage: {version.current_stage}')

version: 4, stage: Staging
version: 5, stage: None


/tmp/ipykernel_29777/2564504774.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  lastest_version = client.get_latest_versions(name=model_name)


In [28]:
model_version = 5
new_stage = 'Production'
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
);

/tmp/ipykernel_29777/4001610271.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [27]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"  
)

<ModelVersion: aliases=[], creation_timestamp=1788173699253, current_stage='Staging', deployment_job_state=None, description='The model version 5 was transitioned to Staging on 2026-08-31', last_updated_timestamp=1788179980581, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='f7577e8748774de6b7b2ee7473bcecaf', run_link=None, source='models:/m-068c811689434d0f9f469f70039004f2', status='READY', status_message=None, tags={}, user_id=None, version=5>

In [ ]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd


def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    # 这段相当于自动拿最新的staging与 发布版相比
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [36]:
df = read_dataframe("/home/dongenxi/mlops-zoomcamp/data/green_tripdata_2021-03.parquet")

In [37]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/home/dongenxi/mlops-zoomcamp/02-experiment-tracking/preprocessor'

In [38]:
import pickle

with open('preprocessor/preprocessor.b', 'rb') as f_in:
    dv = pickle.load(f_in)

In [40]:
X_test = preprocess(df, dv)

target = 'duration'
y_test = df[target].to_numpy()

In [44]:
%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)

CPU times: user 4.84 s, sys: 2.97 s, total: 7.82 s
Wall time: 7.81 s


{'rmse': 6.894249536207721}

In [45]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

CPU times: user 81.4 ms, sys: 70.6 ms, total: 152 ms
Wall time: 149 ms


{'rmse': 6.659623830022514}

In [46]:
client.transition_model_version_stage(
    name=model_name,
    version=4,
    stage="Production",
    archive_existing_versions=True
)

/tmp/ipykernel_29777/3610659809.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1788163908615, current_stage='Production', deployment_job_state=None, description='', last_updated_timestamp=1788183469752, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='', run_link='', source='/home/dongenxi/mlops-zoomcamp/02-experiment-tracking/mlruns/1/models/m-6afb31d76b6945d9bb40a8a807ad4e41/artifacts', status='READY', status_message=None, tags={'model': 'gradientboostingregressor'}, user_id=None, version=4>

In [ ]:
mlflow.register_model()

client.search_runs()